<h1><strong>Project Report: End-to-End Stock Volatility Forecasting and API Deployment</strong></h1>

### Executive Summary

Volatility is a critical metric in financial markets, representing the degree of variation in an asset's price over time and serving as a primary indicator of market risk. For investors, traders, and financial analysts, the ability to accurately forecast volatility is essential for effective risk management, portfolio optimization, and the development of sophisticated trading strategies. This project addresses this need by creating an end-to-end data science solution that predicts daily stock volatility.

We developed a robust system that automates data acquisition from the AlphaVantage financial API, stores the data efficiently in a local SQLite database, and employs a Generalized Autoregressive Conditional Heteroskedasticity (GARCH) model to generate volatility forecasts. The entire pipeline—from data ingestion to prediction—is encapsulated within a deployable FastAPI application. This API exposes two key endpoints: one for on-demand model training with new data and another for generating immediate volatility predictions for any specified stock ticker.

The final product is not just a predictive model but a scalable, production-ready tool. It provides a reliable, programmatic interface for accessing near-term risk forecasts, enabling its integration into larger financial analysis platforms, automated trading systems, or risk management dashboards. This project successfully demonstrates the complete data science lifecycle, delivering a tangible and high-value solution for financial market analysis.

### 1. Introduction

The primary objective of this project was to design, build, and deploy a machine learning system capable of forecasting stock market volatility. Unlike predicting price direction, which is notoriously difficult, forecasting volatility—the magnitude of price fluctuations—is a more tractable problem with significant practical applications. High volatility often signals market uncertainty and increased risk, while low volatility suggests stability.

This project tackles the full scope of creating a data product, encompassing:
1.  **Data Engineering:** Establishing a pipeline to fetch financial data from an external API and store it in a structured, persistent database.
2.  **Exploratory Data Analysis:** Investigating the statistical properties of stock returns to identify patterns, such as volatility clustering, that inform model selection.
3.  **Predictive Modeling:** Implementing and training a GARCH model, a standard econometric tool for modeling financial time series data.
4.  **Model Deployment:** Exposing the model's functionality through a RESTful API, making its predictive power accessible to other applications and users.

The following sections detail the methodology, implementation, and results of each stage of this end-to-end process.

### 2. Data Acquisition and Storage

The foundation of any data science project is a reliable data source. For this project, we utilized the AlphaVantage API to source daily stock market data. To ensure our application is efficient and avoids redundant API calls, we implemented a system to store this data locally in a SQLite database.

#### 2.1. Initial Setup and API Interaction

First, we import the necessary libraries. Sensitive information, such as the API key, is managed securely using a configuration file.

In [ ]:
%pip install arch fastapi pydantic pydantic-settings

import pandas as pd
import requests
from config import settings
import sqlite3
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pydantic import BaseModel

Our first step is to construct a valid URL to query the AlphaVantage API. The URL includes parameters specifying the function (daily time series), the stock ticker symbol, and the API key. Here, we build a request for Ambuja Cement (`AMBUJACEM.BSE`).

In [ ]:
url = (
    "https://learn-api.wqu.edu/1/data-services/alpha-vantage/query?" +
    "function=TIME_SERIES_DAILY&" +
    "symbol=AMBUJACEM.BSE&" +
    f"apikey={settings.alpha_api_key}"
)

print("url type:", type(url))
url

To make the URL more dynamic and accommodate all available API parameters, we define variables for the ticker, output size, and data type.

In [ ]:
ticker = "AMBUJACEM.BSE"
output_size = "compact"
data_type = "json"

url = (
    "https://learn-api.wqu.edu/1/data-services/alpha-vantage/query?" +
    "function=TIME_SERIES_DAILY&" +
    f"symbol={ticker}&" +
    f"output_size={output_size}&" +
    f"data_type={data_type}&" +
    f"apikey={settings.alpha_api_key}"
)

print("url type:", type(url))
url

With a valid URL, we use the `requests` library to send an HTTP GET request to the API. A successful request returns a `200` status code.

In [ ]:
response = requests.get(url=url)

print("response type:", type(response))
print("response status_code:", response.status_code)

The API response contains data in JSON format. We extract this data and load it into a Python dictionary.

In [ ]:
response_data = response.json()

print("response_data type:", type(response_data))

The core time series data is nested within the `'Time Series (Daily)'` key.

In [ ]:
stock_data = response_data['Time Series (Daily)']

print("stock_data type:", type(stock_data))

#### 2.2. Data Transformation and Cleaning

The raw data needs to be transformed into a structured format suitable for analysis. We convert the nested dictionary of stock data into a pandas DataFrame.

In [ ]:
df_ambuja = pd.DataFrame.from_dict(stock_data, orient='index', dtype=float)

print("df_ambuja shape:", df_ambuja.shape)
print()
df_ambuja.info()

The DataFrame's index consists of date strings. For time series analysis, this must be converted to a proper `DatetimeIndex`.

In [ ]:
# Convert `df_ambuja` index to `DatetimeIndex`
df_ambuja.index = pd.to_datetime(df_ambuja.index)

# Name index "date"
df_ambuja.index.name = "date"

df_ambuja.info()

Finally, the column names returned by the API are prefixed with numbers (e.g., `'1. open'`). We clean these names for easier access.

In [ ]:
# Remove numbering from `df_ambuja` column names
df_ambuja.columns = [c.split(". ")[1] for c in df_ambuja.columns]

df_ambuja.info()

#### 2.3. Building a Reusable Data Pipeline

To create a robust and maintainable application, we encapsulate the data fetching and database interaction logic into reusable classes. We use test-driven development principles, writing tests to ensure each component functions as expected.

First, we define the `AlphaVantageAPI` class, which formalizes the process of fetching and cleaning data.

In [ ]:
from data import AlphaVantageAPI

av = AlphaVantageAPI()

In [ ]:
# Define Suzlon ticker symbol
ticker = "SUZLON.BSE"

# Use your `av` object to get daily data
df_suzlon = av.get_daily(ticker=ticker, output_size='full')

print("df_suzlon type:", type(df_suzlon))
print("df_suzlon shape:", df_suzlon.shape)
df_suzlon.head()

Next, we create the `SQLRepository` class to handle all database operations. This class manages the connection to a SQLite database and includes methods for inserting and reading data.

In [ ]:
connection = sqlite3.connect(database=settings.db_name, check_same_thread=False)

print("connection type:", type(connection))

In [ ]:
# Import class definition
from data import SQLRepository

# Create instance of class
repo = SQLRepository(connection=connection)

# Does `repo` have a "connection" attribute?
assert hasattr(repo, "connection")

# Is the "connection" attribute a SQLite `Connection`?
assert isinstance(repo.connection, sqlite3.Connection)

The `insert_table` method writes a DataFrame to a specified table in the database. We test it by inserting the Suzlon data.

In [ ]:
response = repo.insert_table(table_name=ticker, records=df_suzlon, if_exists="replace")

print(response)

The `read_table` method retrieves data from the database, allowing for an optional limit on the number of records. This is crucial for fetching the data needed for model training.

In [ ]:
# Assign `read_table` output to `df_suzlon`
df_suzlon_from_db = repo.read_table(table_name="SUZLON.BSE", limit=2500)

# Is `df_suzlon` a DataFrame?
assert isinstance(df_suzlon_from_db, pd.DataFrame)

print("df_suzlon_from_db shape:", df_suzlon_from_db.shape)
df_suzlon_from_db.head()

### 3. Exploratory Data Analysis

With our data pipeline in place, we perform an exploratory analysis to understand the characteristics of our data. We analyze two stocks: Ambuja Cement (`AMBUJACEM.BSE`), a large, established company, and Suzlon Energy (`SUZLON.BSE`), a company in the renewable energy sector.

First, we load the data for both companies into our database.

In [ ]:
ticker = "AMBUJACEM.BSE"

# Get Ambuja data using `av`
ambuja_records = av.get_daily(ticker=ticker)

# Insert `ambuja_records` into the database using `repo`
response = repo.insert_table(table_name=ticker, records=ambuja_records, if_exists="replace")

print(response)

In [ ]:
df_ambuja_from_db = repo.read_table(table_name="AMBUJACEM.BSE")

print("df_ambuja_from_db type:", type(df_ambuja_from_db))
print("df_ambuja_from_db shape:", df_ambuja_from_db.shape)
df_ambuja_from_db.head()

#### 3.1. Comparing Closing Prices

A simple starting point is to plot the closing prices of both stocks over time.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
# Plot `df_suzlon` and `df_ambuja`
df_suzlon_from_db['close'].plot(ax=ax, label="SUZLON")
df_ambuja_from_db['close'].plot(ax=ax, label='AMBUJA')

# Label axes
plt.xlabel("Date")
plt.ylabel("Closing Price")

# Add legend
plt.legend()

The vast difference in price scales makes a direct comparison difficult. To normalize the data and make it comparable, we analyze daily returns instead of absolute prices. Daily return is the percentage change in the closing price from one day to the next.

In [ ]:
# Sort DataFrames ascending by date before calculating returns
df_ambuja_from_db.sort_index(ascending=True, inplace=True)
df_suzlon_from_db.sort_index(ascending=True, inplace=True)

# Create "return" columns
df_ambuja_from_db['return'] = df_ambuja_from_db["close"].pct_change() * 100
df_suzlon_from_db['return'] = df_suzlon_from_db['close'].pct_change() * 100

df_ambuja_from_db.head()

#### 3.2. Analyzing Daily Returns and Volatility

Plotting the daily returns reveals key characteristics of the stocks' behavior.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
# Plot `df_suzlon` and `df_ambuja` returns
df_suzlon_from_db['return'].plot(ax=ax, label="SUZLON")
df_ambuja_from_db['return'].plot(ax=ax, label='AMBUJA')

# Label axes
plt.xlabel("Date")
plt.ylabel("Daily Return")

# Add legend
plt.legend()

This plot clearly shows that Suzlon's returns are significantly more volatile than Ambuja Cement's. We also observe a phenomenon known as **volatility clustering**: periods of high volatility tend to be followed by more high volatility, and periods of low volatility are followed by more low volatility. This is a stylized fact of financial time series and is the primary motivation for using a GARCH model.

To better visualize this, we can plot the squared returns, which makes the volatility clusters more prominent.

In [ ]:
y_ambuja = df_ambuja_from_db['return'].dropna()

fig, ax = plt.subplots(figsize=(15, 6))

# Plot squared returns
(y_ambuja ** 2).plot(ax=ax)

# Add axis labels
plt.xlabel("Date")
plt.ylabel("Squared Returns")

### 4. Methodology: Volatility Modeling with GARCH

Based on the observed volatility clustering, we selected the Generalized Autoregressive Conditional Heteroskedasticity (GARCH) model. A GARCH model is designed to capture time-varying volatility by modeling the variance of a time series as a function of its past variances and past squared innovations (or shocks).

#### 4.1. Model Specification

To determine the appropriate parameters for our GARCH model, we examine the Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF) plots of the squared returns. These plots help identify the order of the autoregressive (p) and moving average (q) components of the volatility process.